In [1]:
import os
import pickle
import pandas as pd
import numpy as np
import nltk
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi

## Phase 1: Data Recovery and Text Preprocessing
In the original Data Mining pipeline, textual descriptions were dropped to focus on numerical analysis.
Here, the `Description` column is recovered from the raw dataset and merged with the cleaned data.

The natural language text needs to be cleaned by converting it to lowercase, removing punctuation, tokenizing the strings into individual words, and removing stop words.

**Caching note**: `df` is always rebuilt fresh from the source CSVs. The tokenized descriptions, on the other hand, take a minute or so to compute, so they're cached to `data/tokens.pkl` - the same file used by
`8_learning_to_rank.ipynb` for its BM25 section, since both notebooks tokenize with the exact
same `preprocess_text` function.

In [2]:
# Download required NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    return [word for word in tokens if word not in stop_words]

[nltk_data] Downloading package punkt to /home/tommaso/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/tommaso/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/tommaso/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
DATA_DIR = '../data'
TOKENS_PATH = os.path.join(DATA_DIR, 'tokens.pkl')

# 1. df is always rebuilt from the raw CSVs
df_clean = pd.read_csv(os.path.join(DATA_DIR, 'cleaned_dataset.csv'))
df_raw = pd.read_csv(os.path.join(DATA_DIR, 'DM1_game_dataset.csv'))
df_text = df_raw[['BGGId', 'Description']]
df = pd.merge(df_clean, df_text, on='BGGId', how='inner')
df = df.dropna(subset=['Description']).reset_index(drop=True)
df = df.rename(columns={'BGGId': 'ID'})

print(f"Dataset ready! Total board games with descriptions: {len(df)}")

# 2. Tokens: reuse the cache if present and row-count-valid, otherwise compute and persist
tokens_list = None
if os.path.exists(TOKENS_PATH):
    with open(TOKENS_PATH, 'rb') as f:
        cached_tokens = pickle.load(f)
    if len(cached_tokens) == len(df):
        tokens_list = cached_tokens
        print("Reusing cached tokens from disk...")
    else:
        print(
            f"Cache found at '{TOKENS_PATH}' but row count doesn't match df "
            f"({len(cached_tokens)} vs {len(df)}) - treating as stale, will recompute."
        )

if tokens_list is None:
    print("Tokenizing descriptions...")
    tokens_list = df['Description'].apply(preprocess_text).tolist()
    with open(TOKENS_PATH, 'wb') as f:
        pickle.dump(tokens_list, f)
    print(f"Tokens computed and cached to '{TOKENS_PATH}'.")

df['tokens'] = tokens_list

print("Sample tokenization output:")
print(df['tokens'].iloc[0][:10])

Dataset ready! Total board games with descriptions: 19728
Reusing cached tokens from disk...
Sample tokenization output:
['game', 'description', 'publisherin', 'assassins', 'creed', 'arena', 'ndash', 'base', 'assassin', 'creed']


## BM25 Inverted Index Construction
The Okapi BM25 algorithm is used to build an inverted index. BM25 penalizes overly long documents and gives higher weight to rare terms.

In [4]:
# Extract the list of tokenized descriptions
corpus_tokens = df['tokens'].tolist()

# Initialize the BM25 model
bm25 = BM25Okapi(corpus_tokens)

print(f"BM25 Index successfully built for {len(corpus_tokens)} documents!")

BM25 Index successfully built for 19728 documents!


## Query Processing and Retrieval
The following function processes user queries and retrieves the Top-K most relevant board games.

In [5]:
def search_games_bm25(query, top_k=5):
    """
    Searches the BM25 index for the most relevant games given a natural language query.
    """
    # 1. Preprocess the query using the exact same pipeline as the documents
    query_tokens = preprocess_text(query)

    # 2. Get the BM25 scores for all documents in the corpus
    doc_scores = bm25.get_scores(query_tokens)

    # 3. Use a copy of the dataframe to avoid mutating the shared one during testing
    df_temp = df.copy()
    df_temp['bm25_score'] = doc_scores

    # 4. Sort results descending by score and get the top K
    results = df_temp.sort_values(by='bm25_score', ascending=False).head(top_k)

    # Return the Name, Score, and a snippet of the Description
    return results[['Name', 'bm25_score', 'Description']]

### Test the search engine

In [6]:
user_query = "A cooperative space game involving betrayal"
results = search_games_bm25(user_query, top_k=5)

print(f"Results for: '{user_query}'\n")
for i, row in results.iterrows():
    print(f"{row['Name']} (Score: {row['bm25_score']:.2f})")
    print(f"Snippet: {row['Description'][:150]}...\n")

Results for: 'A cooperative space game involving betrayal'

Betrayal at Baldur's Gate (Score: 15.25)
Snippet: description publisherthe shadow bhaal come baldurs gate summon monster horror darknessa build explore iconic city dark alley deadly catacomb work fell...

Cutthroat Caverns (Score: 14.38)
Snippet: quotwithout teamwork survive betrayal yoursquoll winquotcutthroat cavern play   round random encounter   essentially game kill steal round monster enc...

Puerto Diablo (Score: 13.60)
Snippet: setting caribbean   nation try establish island power money goal low level people try survive achieve personal gain possible game people need overcome...

The Binding of Isaac: Four Souls (Score: 13.58)
Snippet: official binding isaac multiplayer card game sacrifice betrayal hoarding player turn play loot card item kill monster yield item loot soul player end ...

SSO (Score: 13.56)
Snippet: semi coop scifi horror survival card counter game player attempt outlast challenge deck narrative threat 

Same query as notebook 7, for a direct BM25-vs-FAISS comparison on identical input. Scores are
BM25 relevance (higher = better match).

Strong on the literal terms it was built for: "betrayal" surfaces directly (*Cutthroat Caverns*,
*Four Souls*), "cooperative" pulls in *SSO*. But "space" barely registers - only *SSO* (sci-fi)
touches the theme, while *Betrayal at Baldur's Gate* and *Puerto Diablo* rank highly purely on
keyword overlap with "betrayal"/"survive", despite being fantasy/pirate settings with no space
element at all. Semantic search in the next notebook is meant to close exactly this kind of gap.